# 02 - Modelos lineares generalizados (Partes 5 a 9)

Ajuste de todas as especificações candidatas à base de 5.570 municípios. Este notebook **compara e não escolhe**:
a escolha do modelo final é feita por validação cruzada no notebook 03, com as mesmas definições de `src/especificacoes.py`.

## Parte 5 - Formulação

**Resposta.** Para o município *i*: Yᵢ = votos de Lula, nᵢ = votos de Lula + votos de Flávio Bolsonaro (denominador do 2º turno),
Yᵢ ~ Bin(nᵢ, πᵢ), g(πᵢ) = xᵢᵀβ. Um único modelo dá os dois candidatos, pois o share de Flávio é 1 − πᵢ.

**Por que o binomial agrupado.** Se cada eleitor do município *i* vota em Lula com probabilidade πᵢ, independentemente dos demais, a verossimilhança
dos nᵢ votos individuais é ∏ πᵢ^yᵢ (1−πᵢ)^(nᵢ−yᵢ). Ela é igual, a menos da constante binomial C(nᵢ, yᵢ), à do binomial agrupado em (yᵢ, nᵢ):
a regressão logística de voto individual com covariáveis **municipais** (todos os eleitores do município compartilham xᵢ) é exatamente o GLM binomial
agrupado. Os pesos nᵢ entram naturalmente: São Paulo pesa milhões de votos, uma cidade pequena pesa centenas.

**Falácia ecológica.** Os coeficientes descrevem como o *share municipal* varia com características *municipais*. Não dizem como um indivíduo evangélico
ou beneficiário do Bolsa Família vota. Interpretamos β como associações entre municípios e usamos o modelo para prever agregados, não para inferir comportamento individual.

**Famílias excluídas.** Gamma e normal inversa têm suporte em (0, ∞) e a Gaussiana em ℝ; a resposta é uma proporção em (0, 1) vinda de contagens com limite superior nᵢ.
Essas famílias não respeitam o suporte nem a relação média-variância πᵢ(1−πᵢ)/nᵢ. Ficam apenas a Gaussiana sobre o logit empírico (linha de base, Parte 7) e as famílias de proporção (binomial, quase-binomial, beta-binomial, beta).

In [1]:
import sys, json, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import stats
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "notebooks"))
from src import especificacoes as E, modelos as M
import graficos_diagnostico as G
pd.options.display.float_format = "{:,.4f}".format
T0 = time.time()
TAB, FIG = ROOT / "results/tables", ROOT / "results/figures"
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / "data/processed/municipios.parquet")
mapa = E.MAPA_2026
y, n = df[mapa["y"]].to_numpy(float), df[mapa["n"]].to_numpy(float)
N = len(df)
print(N, "municípios;", int(y.sum()), "votos Lula de", int(n.sum()), f"= {y.sum()/n.sum():.4%}")
AJ, LINHAS = {}, {}
def ajustar(nome):
    c = E.CANDIDATOS[nome]; a = c.ajustar(df, mapa); AJ[nome] = a
    LINHAS[nome] = E.resumo_ajuste(c, a, df, mapa)
    E.tabela_coeficientes(a).to_csv(TAB / f"coef_{nome}.csv", index_label="termo")
    return a

5570 municípios; 64055133 votos Lula de 123707324 = 51.7796%


**Totais regionais exatos dentro da amostra.** Com a ligação canônica (logit) e dummies de região, as equações de escore do GLM binomial implicam
Σᵢ∈R nᵢ π̂ᵢ = Σᵢ∈R yᵢ para cada região R (as dummies e a constante geram a combinação). O ajuste reproduz os totais regionais *por construção*.
Isso vale só dentro da amostra e **não mede capacidade de previsão regional**: as estimativas regionais devem vir de previsão fora da amostra (notebook 03).

In [2]:
a = ajustar("bin_logit")
pi = E.CANDIDATOS["bin_logit"].prever(a, df, mapa)
reg = pd.DataFrame({"regiao": df[mapa["regiao"]], "observado": y, "ajustado": pi * n}).groupby("regiao").sum()
reg["diferenca"] = reg.ajustado - reg.observado
print(reg.round(4).to_string())
print("máx |diferença| (votos):", reg.diferenca.abs().max(), "| total observado:", reg.observado.sum())

                   observado        ajustado  diferenca
regiao                                                 
Centro-Oeste  3,732,974.0000  3,732,974.0000    -0.0000
Nordeste     24,450,432.0000 24,450,432.0000    -0.0000
Norte         5,024,626.0000  5,024,626.0000    -0.0000
Sudeste      24,037,731.0000 24,037,731.0000    -0.0000
Sul           6,809,370.0000  6,809,370.0000    -0.0000
máx |diferença| (votos): 7.82310962677002e-08 | total observado: 64055133.0


## Parte 6 - Binomial e ligações

**Multicolinearidade.** Calculamos o VIF das covariáveis padronizadas antes de ajustar. A regra do projeto é remover uma covariável de um par só se VIF > 10.

In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
pad = E.Padronizador(mapa).fit(df); X = E.montar_X(df, mapa, pad)
VIF = pd.Series({c: vif(X.values, i) for i, c in enumerate(X.columns) if c != "const"}).sort_values(ascending=False)
VIF.to_frame("VIF").to_csv(TAB / "vif_modelo.csv", index_label="termo")
print(VIF.head(8).round(1).to_string()); print("máx VIF:", round(VIF.max(), 1), "| colunas:", X.shape[1] - 1)

pct_16_24           7.6000
cobertura_pbf       6.2000
lag                 5.8000
pct_60_mais         5.4000
reg_Nordeste        4.9000
pct_pretos_pardos   4.8000
log_dens            3.5000
pct_mulheres        3.5000
máx VIF: 7.6 | colunas: 24


**Decisão:** nenhum VIF passa de 10 (o maior é `pct_16_24`, 7,6), então **nenhuma covariável é removida** por colinearidade. A categoria `bloco_prefeito_2024`
tem só 49 municípios 'centro' e 16 'sem_prefeito'; **fundimos os dois em 'outro'** (65 municípios) para a dummy ter estimativa estável. Referências: bloco 'direita' e região 'Sudeste'.
As contínuas entram em z-score, com média e desvio calculados no treino.

**Ligações binomiais.** Comparamos logit, probit, cloglog, loglog e cauchit com o mesmo preditor linear. Com a mesma resposta e o mesmo *k*, a log-verossimilhança (e portanto AIC/BIC) é diretamente comparável.
Reportamos deviance, deviance/gl e φ de Pearson = Σ(y−nπ̂)²/(nπ̂(1−π̂)) / (N−k); φ ≫ 1 indica sobredispersão.

In [4]:
for nome in ["bin_logit", "bin_probit", "bin_cloglog", "bin_loglog", "bin_cauchit"]:
    ajustar(nome)
cols = ["modelo", "ligacao", "k", "loglik", "aic", "bic", "deviance", "deviance_gl", "phi_pearson", "mae_pond_pp"]
t6 = pd.DataFrame([LINHAS[k] for k in ["bin_logit", "bin_probit", "bin_cloglog", "bin_loglog", "bin_cauchit"]])[cols]
print(t6.to_string(index=False))

     modelo ligacao  k        loglik          aic          bic     deviance  deviance_gl  phi_pearson  mae_pond_pp
  bin_logit   logit 25  -85,965.4073 171,980.8146 172,146.4433 120,006.3172      21.6423      21.6578       1.0584
 bin_probit  probit 25  -87,698.6662 175,447.3324 175,612.9611 123,472.8350      22.2674      22.3089       1.0681
bin_cloglog cloglog 25 -124,443.8150 248,937.6300 249,103.2588 196,963.1327      35.5209      35.9726       1.3326
 bin_loglog  loglog 25 -107,813.5634 215,677.1268 215,842.7555 163,702.6294      29.5226      29.4789       1.2131
bin_cauchit cauchit 25 -106,922.0181 213,894.0362 214,059.6650 161,919.5389      29.2010      28.8724       1.2131


**Leitura.** O logit tem o menor AIC/BIC entre as cinco ligações, de longe. Mas a deviance/gl é da ordem de 20 e o φ de Pearson é ≈ 20: com nᵢ de milhares de votos,
a variância binomial é muito menor que a variação real entre municípios. **Os erros-padrão do binomial puro são grosseiramente pequenos** e os testes não valem; a Parte 7 trata disso.
Escolher a ligação por AIC nesta base ainda é válido para ordenar os links, pois todos sofrem da mesma sobredispersão.

## Parte 7 - Sobredispersão

Quatro tratamentos, mais duas linhas de base:

1. **Quase-binomial (logit)**: mesmos coeficientes do binomial; φ estimado por Pearson (`scale='X2'`) infla os erros-padrão por √φ. Não é uma verossimilhança, então não há AIC/BIC (reportamos QAIC).
2. **Beta-binomial (logit)** (`modelos.BetaBinomial`): πᵢ ~ Beta com precisão θ; Var(Yᵢ) = nᵢπ(1−π)[1 + (nᵢ−1)ρ], ρ = 1/(1+θ). É uma verossimilhança, logo tem AIC/BIC e é a que mais fielmente modela a dispersão extra.
3. **Regressão beta** (`BetaModel`, **sem pesos**): modela a proporção diretamente. Um município de 1.000 eleitores conta como São Paulo, por isso ela não é o objetivo (erro ponderado por votos); fica como comparação.
4. **OLS sobre o logit empírico** log((y+½)/(n−y+½)), sem pesos, e **WLS** com pesos nᵢp̂ᵢ(1−p̂ᵢ) (inverso da variância delta do logit): linhas de base simples.

In [5]:
for nome in ["quasi_logit", "bb_logit", "betareg", "ols_elogit", "wls_elogit"]:
    ajustar(nome)
bb = AJ["bb_logit"].modelo
print(f"quase-binomial: phi = {AJ['quasi_logit'].extra['phi']:.2f}  (erros-padrão x {np.sqrt(AJ['quasi_logit'].extra['phi']):.2f})")
print(f"beta-binomial: theta = {M.theta_de(bb):.1f}, rho = {M.icc(bb):.5f}  (loglik {bb.llf:,.1f})")
lado = pd.concat({"binomial": E.tabela_coeficientes(AJ["bin_logit"])[["coef", "ep"]],
                  "quase-binomial": E.tabela_coeficientes(AJ["quasi_logit"])[["coef", "ep"]],
                  "beta-binomial": E.tabela_coeficientes(AJ["bb_logit"])[["coef", "ep"]]}, axis=1)
lado.to_csv(TAB / "coef_binomial_quasi_bb.csv", index_label="termo")
print(lado.round(3).to_string())

quase-binomial: phi = 21.66  (erros-padrão x 4.65)
beta-binomial: theta = 630.4, rho = 0.00158  (loglik -36,301.6)
                  binomial        quase-binomial        beta-binomial       
                      coef     ep           coef     ep          coef     ep
const               0.1880 0.0010         0.1880 0.0030        0.1950 0.0030
lag                 0.7790 0.0010         0.7790 0.0030        0.7770 0.0030
nota_dep            0.0000 0.0000         0.0000 0.0020        0.0030 0.0020
pct_dep_esq         0.0070 0.0000         0.0070 0.0020        0.0130 0.0020
comparecimento     -0.0130 0.0000        -0.0130 0.0020       -0.0180 0.0020
delta_comp         -0.0080 0.0000        -0.0080 0.0010       -0.0030 0.0010
log_pop             0.0050 0.0000         0.0050 0.0010        0.0060 0.0020
log_dens            0.0210 0.0000         0.0210 0.0010        0.0340 0.0020
pct_mulheres        0.0330 0.0000         0.0330 0.0020        0.0270 0.0020
pct_16_24          -0.0240 0.0010     

In [6]:
rng = np.random.default_rng(0)
pis = {k: E.CANDIDATOS[k].prever(AJ[k], df, mapa) for k in ["bin_logit", "bb_logit"]}
rq = {"binomial": M.residuos_quantilicos(y, n, pis["bin_logit"], rng=np.random.default_rng(1)),
      "beta-binomial": M.residuos_quantilicos(y, n, pis["bb_logit"], theta=M.theta_de(bb), rng=np.random.default_rng(1))}
G.salvar(G.qq_normal(rq), FIG / "mlg_qq_binomial_vs_bb.html")
G.salvar(G.forest_coeficientes({"binomial": E.tabela_coeficientes(AJ["bin_logit"]), "beta-binomial": E.tabela_coeficientes(AJ["bb_logit"])}),
         FIG / "mlg_forest_binomial_vs_bb.html")
print({k: (round(v.mean(), 3), round(v.std(), 3)) for k, v in rq.items()})

{'binomial': (np.float64(-0.008), np.float64(3.426)), 'beta-binomial': (np.float64(-0.004), np.float64(1.004))}


**Leitura.** Os coeficientes pontuais mudam pouco entre binomial e quase-binomial (são idênticos), mas os erros-padrão do quase-binomial são ≈ √φ vezes maiores; no beta-binomial os pesos
deixam de ser proporcionais a nᵢ (variância ≈ nᵢπ(1−π)[1+(nᵢ−1)ρ] satura em ρ), então municípios grandes deixam de dominar o ajuste e os coeficientes e SEs mudam.
Os resíduos quantílicos (QQ-plot em `results/figures/mlg_qq_binomial_vs_bb.html`) devem ter média ≈ 0 e desvio ≈ 1 se o modelo descreve a dispersão: o binomial puro fica com desvio muito maior que 1.

## Parte 8 - Preditor linear

Variamos o preditor linear sobre o beta-binomial (verossimilhança) e sobre o quase-binomial (mesmo ponto de partida do binomial, SE robustos à dispersão):

- **Offset do lag (modelo de swing):** coeficiente de `logit_pt_2t_2022` fixo em 1, isto é, logit(πᵢ) = logit(share 2022) + xᵢᵀβ. Testa-se contra o coeficiente livre.
- **Spline no lag:** B-spline cúbica com 4 graus de liberdade (`patsy.bs`); valores do teste fora do suporte do treino são limitados à fronteira.
- **Região × lag:** inclinação do lag diferente por região.
- **Efeitos fixos de UF** no lugar da região.
- **GLMM** com intercepto aleatório por UF.
- **Seleção backward por BIC** no beta-binomial.
- **Regressão logística L1** (laço) no fractional logit municipal.

In [7]:
for nome in ["bb_offset", "quasi_offset", "bb_spline", "quasi_spline", "bb_reglag", "quasi_reglag", "bb_uf", "quasi_uf"]:
    ajustar(nome)
# coeficiente livre do lag (escala bruta = coef_z / dp)
dp = AJ["bb_logit"].pad.desvio["lag"]; tb = E.tabela_coeficientes(AJ["bb_logit"]).loc["lag"]
b_raw, ep_raw = tb.coef / dp, tb.ep / dp
print(f"lag livre (beta-binomial): coef bruto = {b_raw:.3f} (EP {ep_raw:.3f}); H0: coef=1 -> z = {(b_raw-1)/ep_raw:.2f}")
tq = E.tabela_coeficientes(AJ["quasi_logit"]).loc["lag"]; bq = tq.coef / dp
print(f"lag livre (quase-binomial): coef bruto = {bq:.3f} (EP {tq.ep/dp:.3f}); z(H0=1) = {(bq-1)/(tq.ep/dp):.2f}")
lr = 2 * (AJ["bb_logit"].loglik - AJ["bb_offset"].loglik)
print(f"LRT offset vs livre (beta-binomial): 2*dlogL = {lr:.1f}, gl=1, p = {stats.chi2.sf(lr, 1):.2e}; dAIC(offset - livre) = {-2*AJ['bb_offset'].loglik+2*AJ['bb_offset'].k - (-2*AJ['bb_logit'].loglik+2*AJ['bb_logit'].k):.1f}")
for a_, b_ in [("bb_logit", "bb_spline"), ("bb_logit", "bb_reglag")]:
    lr2 = 2 * (AJ[b_].loglik - AJ[a_].loglik); gl = AJ[b_].k - AJ[a_].k
    print(f"LRT {a_} vs {b_}: 2*dlogL = {lr2:.1f}, gl={gl}, p = {stats.chi2.sf(lr2, gl):.2e}")
print("quase-binomial (teste F de Wald na deviance escalada):")
for a_, b_ in [("quasi_logit", "quasi_offset"), ("quasi_logit", "quasi_spline"), ("quasi_logit", "quasi_reglag"), ("quasi_logit", "quasi_uf")]:
    da = LINHAS[a_]["deviance"]; db = LINHAS[b_]["deviance"]; ph = AJ["quasi_logit"].extra["phi"]
    gl = abs(AJ[b_].k - AJ[a_].k)
    print(f"  {a_} -> {b_}: |ddeviance| = {abs(da-db):.1f}, gl={gl}, F = {abs(da-db)/gl/ph:.2f}, p = {stats.f.sf(abs(da-db)/gl/ph, gl, N-AJ[b_].k):.2e}")

lag livre (beta-binomial): coef bruto = 0.899 (EP 0.004); H0: coef=1 -> z = -28.48
lag livre (quase-binomial): coef bruto = 0.901 (EP 0.003); z(H0=1) = -29.48
LRT offset vs livre (beta-binomial): 2*dlogL = 754.7, gl=1, p = 3.76e-166; dAIC(offset - livre) = 752.7
LRT bb_logit vs bb_spline: 2*dlogL = 31.4, gl=3, p = 6.95e-07
LRT bb_logit vs bb_reglag: 2*dlogL = 58.0, gl=4, p = 7.56e-12
quase-binomial (teste F de Wald na deviance escalada):
  quasi_logit -> quasi_offset: |ddeviance| = 18745.3, gl=1, F = 865.52, p = 6.60e-177
  quasi_logit -> quasi_spline: |ddeviance| = 360.1, gl=3, F = 5.54, p = 8.52e-04
  quasi_logit -> quasi_reglag: |ddeviance| = 988.4, gl=4, F = 11.41, p = 3.19e-09
  quasi_logit -> quasi_uf: |ddeviance| = 20863.4, gl=22, F = 43.79, p = 5.92e-174


In [8]:
t8 = pd.DataFrame([LINHAS[k] for k in ["bb_logit", "bb_offset", "bb_spline", "bb_reglag", "bb_uf",
                                         "quasi_logit", "quasi_offset", "quasi_spline", "quasi_reglag", "quasi_uf"]])[cols + ["qaic"]]
print(t8.to_string(index=False))
sp = AJ["bb_spline"]; ts = E.tabela_coeficientes(sp)
print(ts.filter(like="lag_bs", axis=0).round(3).to_string())
tr_ = E.tabela_coeficientes(AJ["bb_reglag"]); print(tr_.filter(like=":lag", axis=0).round(3).to_string())

      modelo ligacao  k       loglik         aic         bic     deviance  deviance_gl  phi_pearson  mae_pond_pp       qaic
    bb_logit   logit 26 -36,301.5522 72,655.1044 72,827.3584 131,674.5844      23.7508      23.7634       1.1665        NaN
   bb_offset   logit 25 -36,678.9162 73,407.8325 73,573.4613 169,941.5149      30.6477      30.8143       1.3717        NaN
   bb_spline   logit 29 -36,285.8453 72,629.6905 72,821.8199 133,404.5125      24.0759      24.0939       1.1754        NaN
   bb_reglag   logit 30 -36,272.5426 72,605.0852 72,803.8397 131,622.4367      23.7586      23.7958       1.1505        NaN
       bb_uf   logit 48 -35,895.1520 71,886.3040 72,204.3112 125,642.3326      22.7530      22.7529       1.1815        NaN
 quasi_logit   logit 26          NaN         NaN         NaN 120,006.3172      21.6462      21.6617       1.0584 7,990.5224
quasi_offset   logit 25          NaN         NaN         NaN 138,751.6119      25.0228      25.1719       1.1504 7,626.3194
quasi_sp

**GLMM.** `BinomialBayesMixedGLM` do statsmodels exige resposta 0/1 e não aceita (y, n) agrupados. Mostramos a falha e usamos um GLMM por PQL próprio
(`ModeloGLMM` em `especificacoes.py`): intercepto aleatório por UF, variância por EM, pesos divididos por φ (equivale a um GLMM quase-binomial).

In [9]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
Xg = E.montar_X(df, mapa, pad, E.Termos(usa_uf=True, uf_aleatoria=True))
try:
    BinomialBayesMixedGLM(y / n, Xg.filter(regex="^(?!uf_)"), Xg.filter(like="uf_").to_numpy(), np.zeros(27, int))
    print("BinomialBayesMixedGLM aceitou a resposta fracionária (inesperado)")
except Exception as e:
    print("BinomialBayesMixedGLM recusou a resposta agrupada/fracionária (exige endog 0/1):", type(e).__name__)
ajustar("glmm_uf")
g = AJ["glmm_uf"]; print({k: round(v, 4) for k, v in g.extra.items()})
ef = pd.Series(g.modelo.beta, index=g.nomes).filter(like="uf_").sort_values()
print("efeitos aleatórios de UF (logit): menores", ef.head(3).round(3).to_dict(), "maiores", ef.tail(3).round(3).to_dict())

BinomialBayesMixedGLM recusou a resposta agrupada/fracionária (exige endog 0/1): ValueError
{'sigma_uf': 0.0532, 'phi': 17.8975}
efeitos aleatórios de UF (logit): menores {'uf_GO': -0.095, 'uf_RS': -0.076, 'uf_CE': -0.059} maiores {'uf_SE': 0.082, 'uf_AP': 0.085, 'uf_AC': 0.094}


In [10]:
t0 = time.time()
vs, trilha = E.selecao_backward_bic(df, mapa)
removidas = [v for v in E.CONTINUAS if v not in vs]
print(f"backward por BIC ({time.time()-t0:.0f}s): mantidas {len(vs)} contínuas; removidas {removidas}")
print(trilha.round(1).to_string(index=False))
E.registrar_backward(vs); ajustar("bb_backward")
ajustar("l1_logit")
l1 = AJ["l1_logit"]; kept = [c for c in l1.extra["mantidas"]]; zeros = [c for c in l1.nomes if c not in kept]
print(f"L1: C(1 EP) = {l1.extra['C']:.4g}, C(mín. CV) = {l1.extra['C_min']:.4g}; mantidas {len(kept)}/{len(l1.nomes)}: {kept}\nzeradas: {zeros}")
json.dump({"backward_bic": vs, "backward_removidas": removidas, "l1_mantidas": kept, "l1_zeradas": zeros, "l1_C": l1.extra["C"], "l1_C_min": l1.extra["C_min"]},
          open(TAB / "selecao_variaveis.json", "w"), indent=1, ensure_ascii=False)

backward por BIC (14s): mantidas 12 contínuas; removidas ['nota_dep', 'delta_comp', 'log_pop', 'pct_60_mais', 'pib_pc', 'pct_urbana']
   removida         bic  n_vars
        NaN 72,827.4000      18
pct_60_mais 72,819.6000      17
     pib_pc 72,812.3000      16
   nota_dep 72,806.2000      15
 delta_comp 72,800.8000      14
 pct_urbana 72,797.5000      13
    log_pop 72,794.9000      12


L1: C(1 EP) = 0.03162, C(mín. CV) = 100; mantidas 6/24: ['lag', 'log_dens', 'pct_mulheres', 'cobertura_pbf', 'pct_vab_agro', 'pct_pretos_pardos']
zeradas: ['nota_dep', 'pct_dep_esq', 'comparecimento', 'delta_comp', 'log_pop', 'pct_16_24', 'pct_60_mais', 'pct_superior', 'pct_analfabeto', 'pib_pc', 'pct_evangelicos', 'pct_urbana', 'bloco_esquerda', 'bloco_outro', 'reg_Centro-Oeste', 'reg_Nordeste', 'reg_Norte', 'reg_Sul']


**Leitura da Parte 8.** O backward por BIC remove apenas contínuas (o lag, o bloco do prefeito e a região ficam sempre; critério: remover a contínua de menor |z| enquanto o BIC cair).
No L1, C é escolhido por validação cruzada de 5 grupos (regra de 1 erro-padrão: o menor C, isto é, o laço mais forte, cuja perda fica a 1 EP do mínimo; perda = log-loss fracionária, cada município com peso 1: duas linhas com peso p e 1−p, X padronizado); o laço
aqui **não pondera por votos**, é uma referência de parcimônia. As listas de variáveis mantidas e zeradas estão em `results/tables/selecao_variaveis.json`.

## Parte 9 - Comparação

Todas as linhas são **dentro da amostra**. AIC/BIC só valem para modelos com verossimilhança binomial sobre (y, n): binomial, beta-binomial e (aproximadamente, sem penalização) L1.
Quase-binomial e GLMM-PQL não têm verossimilhança (QAIC = −2ℓ/φ + 2k aparece só para o quase-binomial); regressão beta, OLS e WLS têm outra resposta, então AIC/BIC ficam `NaN`.
O erro MAE ponderado (p.p.) dentro da amostra **favorece modelos mais flexíveis** e não substitui a validação cruzada.

In [11]:
for nome in E.CANDIDATOS:
    if nome not in AJ: ajustar(nome)
comp = E.tabela_comparacao(LINHAS)
comp.to_csv(TAB / "comparacao_modelos.csv", index=False)
print(comp[["modelo", "familia", "ligacao", "k", "aic", "bic", "qaic", "deviance_gl", "phi_pearson", "mae_pond_pp"]].to_string(index=False))

      modelo  familia    ligacao  k          aic          bic       qaic  deviance_gl  phi_pearson  mae_pond_pp
       bb_uf  betabin      logit 48  71,886.3040  72,204.3112        NaN      22.7530      22.7529       1.1815
   bb_reglag  betabin      logit 30  72,605.0852  72,803.8397        NaN      23.7586      23.7958       1.1505
   bb_spline  betabin      logit 29  72,629.6905  72,821.8199        NaN      24.0759      24.0939       1.1754
    bb_logit  betabin      logit 26  72,655.1044  72,827.3584        NaN      23.7508      23.7634       1.1665
 bb_backward  betabin      logit 20  72,662.3500  72,794.8530        NaN      23.7643      23.7809       1.1560
   bb_offset  betabin      logit 25  73,407.8325  73,573.4613        NaN      30.6477      30.8143       1.3717
   bin_logit binomial      logit 25 171,980.8146 172,146.4433        NaN      21.6423      21.6578       1.0584
  bin_probit binomial     probit 25 175,447.3324 175,612.9611        NaN      22.2674      22.3089      

In [12]:
bbs = comp[comp.familia == "betabin"].set_index("modelo")
print((bbs[["aic", "bic"]] - bbs[["aic", "bic"]].min()).round(1).sort_values("bic").to_string())
print(f"tempo total do notebook: {time.time()-T0:.0f}s")

                   aic        bic
modelo                           
bb_uf           0.0000     0.0000
bb_backward   776.0000   590.5000
bb_reglag     718.8000   599.5000
bb_spline     743.4000   617.5000
bb_logit      768.8000   623.0000
bb_offset   1,521.5000 1,369.2000
tempo total do notebook: 52s


### Candidatos para a validação cruzada (notebook 03)

Não escolhemos o modelo final aqui. Os números acima são dentro da amostra; o notebook 03 decide por validação cruzada (k-fold e, para as previsões por UF, leave-one-UF-out). Levamos **cinco candidatos**, mais uma linha de base:

| Chave em `CANDIDATOS` | Por que entra |
|---|---|
| `bb_reglag` | Menor AIC entre os beta-binomiais sem efeito de UF; a inclinação do lag varia por região (LRT p ≈ 1e-11). Verossimilhança completa (permite intervalos). |
| `bb_backward` | Menor BIC entre os modelos sem UF (12 contínuas): a opção parcimoniosa. Registrar com `E.registrar_backward(vs)` usando `selecao_variaveis.json` (`backward_bic`) antes de chamar. |
| `bb_uf` | Menor AIC/BIC de todos (UF fixa melhora ≈ 770 de AIC sobre a região). Só serve em k-fold: no leave-one-UF-out o nível do estado de teste não existe e as dummies viram a referência (SP). |
| `quasi_reglag` | Pesa os municípios por votos (como o erro nacional), com o menor erro ponderado dentro da amostra entre os modelos sem UF (1,05 p.p.), e φ ≈ 21,5 corrige os EPs. Sem AIC; a CV decide. |
| `glmm_uf` | Intercepto aleatório por UF (σ ≈ 0,05 no logit): encolhe estados pequenos e aceita UF nova (efeito 0). É o candidato com UF válido no leave-one-UF-out. |

Linha de base: `bb_logit` (spec principal) e `quasi_logit`. Descartados: ligações probit/cloglog/loglog/cauchit (AIC muito pior que o logit), `*_offset` (offset rejeitado; ver abaixo), regressão beta e OLS/WLS (outra resposta, não ponderam por votos como queremos), `l1_logit` (o laço do fractional logit não pondera por votos e perde muito de ajuste).

**Conclusões da Parte 8 e 9.** (i) O offset do lag (swing puro) é rejeitado: o coeficiente livre do lag é ≈ 0,90 (z = −28 contra 1) e o AIC piora ≈ 750. Os municípios regridem à média em relação a 2022. (ii) A spline do lag e a interação região × lag melhoram a verossimilhança de modo significativo, mas pouco no erro ponderado. (iii) UF fixa melhora muito a verossimilhança; no erro ponderado, o quase-binomial com UF cai de 1,06 para 0,92 p.p. (iv) O beta-binomial dá θ ≈ 630 e ρ ≈ 0,0016: correlação intraclasse pequena, mas com nᵢ de milhares, 1 + (n−1)ρ é da ordem de 20 (a mesma sobredispersão do φ de Pearson no binomial). Os φ e a deviance/gl do beta-binomial são calculados com a variância binomial e **não** medem o ajuste dele: use o QQ-plot dos resíduos quantílicos (desvio ≈ 1,00 contra 3,4 no binomial).

## Parte 10 - Diagnóstico

Até aqui comparamos modelos por AIC/BIC dentro da amostra. Esta parte verifica se os **pressupostos** de cada candidato se sustentam, para que a escolha (feita por validação cruzada no notebook 03) não repouse em uma suposição falsa. Modelos diagnosticados: `bin_logit`, `quasi_reglag`, `bb_logit`, `bb_reglag`, `bb_spline` e `glmm_uf`.

Pressupostos e como cada verificação os testa:

1. **Distribuição da resposta** (binomial, ou beta-binomial): resíduos quantílicos aleatorizados (Dunn & Smyth, 1996) devem ser N(0,1); QQ-plot e gráfico half-normal com envelope simulado.
2. **Ausência de pontos que dominam o ajuste**: alavanca e distância de Cook; reajuste sem os 10 municípios mais influentes.
3. **Forma do preditor e variância**: resíduo contra o ajustado, contra `lag` (resultado de 2022) e por região.
4. **Independência entre municípios**: I de Moran dos resíduos (999 permutações) e mapa de resíduos.

**Resíduos quantílicos.** Para cada município, sorteamos u ~ U(F(y-1), F(y)), em que F é a função de distribuição ajustada, e tomamos r = Φ⁻¹(u). Para resposta discreta, o resíduo de Pearson ou o de deviance não é normal nem com o modelo certo; o quantílico é. Para a binomial usamos F binomial com π̂ᵢ e nᵢ; para a beta-binomial, a beta-binomial com θ̂. Os modelos quase-binomial e GLMM não têm verossimilhança completa; para eles os resíduos são os da binomial com o π̂ᵢ do modelo. Logo, um desvio padrão acima de 1 nesses dois mede a sobredispersão que eles absorvem apenas nos erros-padrão (quase) ou no efeito de UF (GLMM).

In [13]:
t10 = time.time()
MODELOS_D = ["bin_logit", "quasi_reglag", "bb_logit", "bb_reglag", "bb_spline", "glmm_uf"]
for nome in MODELOS_D:
    if nome not in AJ: ajustar(nome)
PI, TH, R = {}, {}, {}
for nome in MODELOS_D:
    c = E.CANDIDATOS[nome]; PI[nome] = np.asarray(c.prever(AJ[nome], df, mapa), float)
    TH[nome] = AJ[nome].extra.get("theta") if c.familia == "betabin" else None
    R[nome] = G.residuos_quantilicos_rapido(y, n, PI[nome], theta=TH[nome], rng=0)
# conferência da quadratura contra a FDA exata da scipy (mesmos sorteios uniformes): 100 maiores municípios + 1 a cada 30
sub = np.unique(np.r_[np.argsort(-n)[:100], np.arange(0, N, 30)])
r_ex = M.residuos_quantilicos(y[sub], n[sub], PI["bb_reglag"][sub], theta=TH["bb_reglag"], rng=0)
r_ap = G.residuos_quantilicos_rapido(y[sub], n[sub], PI["bb_reglag"][sub], theta=TH["bb_reglag"], rng=0)
print(f"quadratura x scipy ({len(sub)} mun.): máx |dif| = {np.abs(r_ex - r_ap).max():.4f}; percentil 99 = {np.quantile(np.abs(r_ex - r_ap), .99):.5f}")
reg_s = df[mapa["regiao"]].astype(str).to_numpy()
resumo = pd.DataFrame({nome: {"media": R[nome].mean(), "dp": R[nome].std(ddof=1),
                              "assimetria": stats.skew(R[nome]), "curtose_exc": stats.kurtosis(R[nome]),
                              "pct_abs_gt2": 100 * np.mean(np.abs(R[nome]) > 2),
                              "pct_abs_gt3": 100 * np.mean(np.abs(R[nome]) > 3)} for nome in MODELOS_D}).T
resumo.index.name = "modelo"; resumo.to_csv(TAB / "diag_residuos_resumo.csv")
pd.DataFrame(R).assign(id_municipio=df["id_municipio"].to_numpy()).to_csv(TAB / "diag_residuos_quantilicos.csv", index=False)
print(resumo.round(3).to_string())
print("esperado sob N(0,1): dp 1, assimetria 0, curtose 0, |r|>2: 4,55%, |r|>3: 0,27%")

quadratura x scipy (283 mun.): máx |dif| = 1.9008; percentil 99 = 0.03692
               media     dp  assimetria  curtose_exc  pct_abs_gt2  pct_abs_gt3
modelo                                                                        
bin_logit    -0.0080 3.4270     -0.0290      -0.6050      55.0090      38.4920
quasi_reglag  0.0150 3.4220     -0.0420      -0.6020      55.3680      38.6710
bb_logit     -0.0070 1.0340     -0.4310       3.9330       5.0090       1.0590
bb_reglag    -0.0030 1.0330     -0.4120       3.8230       4.8650       1.0950
bb_spline    -0.0040 1.0330     -0.4360       3.8360       4.9730       1.0950
glmm_uf      -0.1090 3.2540      0.0380      -0.4380      51.9750      35.0450
esperado sob N(0,1): dp 1, assimetria 0, curtose 0, |r|>2: 4,55%, |r|>3: 0,27%


### Passo 1 - QQ-plot dos resíduos quantílicos

Se a distribuição da resposta estiver certa, os pontos ficam sobre a diagonal. Inclinação maior que 1 indica variância subestimada (sobredispersão); curvatura nas caudas indica caudas mais pesadas que o modelo. Cada curva usa no máximo 1.500 pontos (a figura completa está em `results/figures/diag_qq.html`).

In [14]:
fig = G.qq_normal(R, "QQ-plot dos resíduos quantílicos (N(0,1) esperado)")
G.salvar(fig, FIG / "diag_qq.html"); fig.show()

### Passo 2 - Half-normal com envelope simulado

Gráfico de |resíduo| ordenado contra quantis half-normal, com banda de 95% obtida por **bootstrap paramétrico**: simulamos a resposta sob o modelo ajustado, reajustamos o modelo inteiro, recalculamos os resíduos e tomamos os quantis 2,5% e 97,5% ponto a ponto (Atkinson, 1985; Moral et al., 2017). O envelope por reajuste leva em conta que os parâmetros foram estimados dos mesmos dados.

**Decisão de custo:** `bin_logit` usa B = 99 (cada reajuste leva ~0,04 s). `bb_reglag` usa **B = 49**, porque cada reajuste beta-binomial leva ~4 s; com 49 simulações os extremos da banda são o 2º menor e o 2º maior valor simulados (nível nominal ~96%) e a banda é mais ruidosa. Pontos fora da banda são contados ponto a ponto, então uma fração de alguns por cento fora já é esperada mesmo com o modelo certo.

Os resíduos beta-binomiais usam uma quadratura da FDA (`G.residuos_quantilicos_rapido`): a `scipy.stats.betabinom.cdf` leva ~9 s por chamada, o que tornaria cada reajuste do envelope três vezes mais lento. A conferência no passo 1 mostra que 99% dos resíduos diferem menos de 0,04 da versão exata; as diferenças maiores ficam nas caudas extremas (|r| > 4), onde um erro de 1e-4 na FDA se amplifica em Φ⁻¹, e não alteram nenhuma conclusão.

In [15]:
# os resíduos de cada reajuste usam o y simulado daquele reajuste (estado['y'])
def env_modelo(nome, B, seed):
    c = E.CANDIDATOS[nome]; col_y = mapa["y"]; estado = {"y": y}
    def ajustar_fn(ys=None):
        estado["y"] = y if ys is None else ys
        return c.ajustar(df if ys is None else df.assign(**{col_y: ys}), mapa)
    def residuos_fn(res):
        return G.residuos_quantilicos_rapido(estado["y"], n, c.prever(res, df, mapa),
                                      theta=res.extra.get("theta") if c.familia == "betabin" else None, rng=0)
    def simular_fn(res, rng):
        pi_ = c.prever(res, df, mapa)
        if c.familia == "betabin":
            th = res.extra["theta"]; pi_ = rng.beta(pi_ * th, (1 - pi_) * th)
        return rng.binomial(n.astype(int), pi_).astype(float)
    return M.envelope_meio_normal(residuos_fn, simular_fn, ajustar_fn, B=B, rng=seed)

ENV, lin = {}, []
for nome, B in [("bin_logit", 99), ("bb_reglag", 49)]:
    t0 = time.time(); ENV[nome] = env_modelo(nome, B, 2026); e = ENV[nome]
    fora = (e["observado"] > e["superior"]) | (e["observado"] < e["inferior"])
    acima = e["observado"] > e["superior"]
    lin.append({"modelo": nome, "B": B, "pct_fora": 100 * fora.mean(), "pct_acima": 100 * acima.mean(), "pct_abaixo": 100 * (e["observado"] < e["inferior"]).mean(),
                "maior_razao_obs_sup": float((e["observado"] / e["superior"]).max()), "segundos": time.time() - t0})
    pd.DataFrame({k: e[k] for k in ["teorico", "observado", "inferior", "mediana", "superior"]}).to_csv(TAB / f"diag_envelope_{nome}.csv", index=False)
env_res = pd.DataFrame(lin); env_res.to_csv(TAB / "diag_envelope_resumo.csv", index=False)
print(env_res.round(3).to_string())
fig = G.meio_normal(ENV, "Half-normal com envelope simulado de 95% (bin_logit B=99; bb_reglag B=49)")
G.salvar(fig, FIG / "diag_meio_normal.html"); fig.show()

      modelo   B  pct_fora  pct_acima  pct_abaixo  maior_razao_obs_sup  segundos
0  bin_logit  99   99.9640    99.9640      0.0000               4.0580    6.1800
1  bb_reglag  49   93.8780     4.6500     89.2280               1.7450  363.0410


### Passo 3 - Alavanca e distância de Cook

A alavanca hᵢ mede quanto o município i puxa o próprio ajuste (diagonal da matriz-chapéu ponderada W^½X(X'WX)⁻¹X'W^½ᵀ, com W = nᵢπ̂ᵢ(1-π̂ᵢ)). A distância de Cook combina alavanca e resíduo de Pearson: Dᵢ = rᵢ² hᵢ / (φ̂ p (1-hᵢ)²). Usamos o `quasi_reglag` (GLM quase-binomial, φ̂ de Pearson). Calculamos à mão, porque o `get_influence()` do statsmodels não trata a resposta de duas colunas com φ de Pearson própria. Valores de referência convencionais: h > 2p/N e D > 4/N, que num N grande sinalizam muitos municípios e servem só como triagem; o que importa é o efeito do reajuste sem os 10 maiores.

Como os pesos da binomial crescem com nᵢ, os municípios grandes (capitais) tendem a ter alavanca alta.

In [16]:
a_q = AJ["quasi_reglag"]; cq = E.CANDIDATOS["quasi_reglag"]
Xq = E.montar_X(df, mapa, a_q.pad, cq.termos).to_numpy(float); pq = Xq.shape[1]; phi_q = a_q.extra["phi"]
pi_q = PI["quasi_reglag"]; W = n * pi_q * (1 - pi_q)
A = np.linalg.inv(Xq.T @ (W[:, None] * Xq))
h = W * np.einsum("ij,jk,ik->i", Xq, A, Xq)
rP = (y - n * pi_q) / np.sqrt(W)
cook = rP ** 2 * h / (phi_q * pq * (1 - h) ** 2)
infl = df[["id_municipio", "nome_municipio", "sigla_uf"]].assign(n=n, prop_obs=y / n, prop_aj=pi_q, alavanca=h, cook=cook, pearson=rP)
top10 = infl.sort_values("cook", ascending=False).head(10)
top10.to_csv(TAB / "diag_influencia_top10.csv", index=False)
print(f"p = {pq}; phi = {phi_q:.2f}; alavanca média = {h.mean():.4f}; max = {h.max():.3f}; "
      f"h > 2p/N: {(h > 2*pq/N).sum()} mun.; D > 4/N: {(cook > 4/N).sum()} mun.; max D = {cook.max():.4f}")
print(top10.round(4).to_string(index=False))
fig = G.alavanca_cook(h, cook, (df["nome_municipio"].astype(str) + "/" + df["sigla_uf"].astype(str)).tolist(), pq)
G.salvar(fig, FIG / "diag_alavanca_cook.html"); fig.show()

p = 29; phi = 21.49; alavanca média = 0.0052; max = 0.672; h > 2p/N: 481 mun.; D > 4/N: 424 mun.; max D = 5.1226
 id_municipio   nome_municipio sigla_uf              n  prop_obs  prop_aj  alavanca   cook  pearson
      5300108         Brasília       DF 1,804,766.0000    0.4572   0.4489    0.6718 5.1226  22.6254
      3550308        São Paulo       SP 6,962,277.0000    0.5550   0.5606    0.5558 3.9315 -29.4922
      1302603           Manaus       AM 1,178,310.0000    0.4417   0.4242    0.2565 1.1003  38.4495
      2304400        Fortaleza       CE 1,551,916.0000    0.6007   0.6120    0.2799 0.7180 -28.7932
      2611606           Recife       PE 1,033,711.0000    0.6127   0.5984    0.1843 0.3941  29.7750
      5208707          Goiânia       GO   805,038.0000    0.3736   0.3886    0.1657 0.2904 -27.5682
      3304557   Rio de Janeiro       RJ 3,683,589.0000    0.5154   0.5119    0.3498 0.2330  13.2506
      1501402            Belém       PA   852,348.0000    0.5501   0.5419    0.1834 0.1

In [17]:
# Reajuste sem os 10 municípios mais influentes; mudança em unidades de EP do ajuste completo.
# A padronização (z-score) é refeita no subconjunto, o que desloca os coeficientes de forma desprezível (dp muda ~1e-4 em termos relativos).
df_s = df.drop(index=top10.index)
linhas = []
for nome in ["quasi_reglag", "bb_reglag"]:
    c = E.CANDIDATOS[nome]; a_s = c.ajustar(df_s, mapa)
    t_c, t_s = E.tabela_coeficientes(AJ[nome]), E.tabela_coeficientes(a_s)
    comum = [i for i in t_c.index if i in t_s.index]
    d = pd.DataFrame({"modelo": nome, "termo": comum, "coef_completo": t_c.loc[comum, "coef"].to_numpy(),
                      "coef_sem10": t_s.loc[comum, "coef"].to_numpy(), "ep_completo": t_c.loc[comum, "ep"].to_numpy()})
    d["mudanca_em_ep"] = (d.coef_sem10 - d.coef_completo) / d.ep_completo
    linhas.append(d)
sem10 = pd.concat(linhas, ignore_index=True); sem10.to_csv(TAB / "diag_sem_influentes.csv", index=False)
for nome, d in sem10.groupby("modelo"):
    j = d.mudanca_em_ep.abs().idxmax()
    print(f"{nome}: maior |mudança| = {d.mudanca_em_ep[j]:+.2f} EP em '{d.termo[j]}'; termos com |mudança| > 0,5 EP: {(d.mudanca_em_ep.abs() > 0.5).sum()} de {len(d)}; "
          f"termos que trocam de sinal: {(np.sign(d.coef_completo) != np.sign(d.coef_sem10)).sum()}")
print(sem10.assign(a=sem10.mudanca_em_ep.abs()).sort_values("a", ascending=False).head(6).drop(columns="a").round(4).to_string(index=False))

bb_reglag: maior |mudança| = -0.14 EP em 'bloco_outro'; termos com |mudança| > 0,5 EP: 0 de 30; termos que trocam de sinal: 0
quasi_reglag: maior |mudança| = -8.43 EP em 'bloco_outro'; termos com |mudança| > 0,5 EP: 22 de 29; termos que trocam de sinal: 2
      modelo       termo  coef_completo  coef_sem10  ep_completo  mudanca_em_ep
quasi_reglag bloco_outro         0.0756      0.0240       0.0061        -8.4337
quasi_reglag  delta_comp        -0.0077     -0.0016       0.0014         4.5136
quasi_reglag   reg_Norte         0.0462      0.0232       0.0064        -3.5670
quasi_reglag     reg_Sul        -0.0325     -0.0527       0.0061        -3.3294
quasi_reglag         lag         0.7712      0.7844       0.0042         3.1321
quasi_reglag reg_Sul:lag         0.0128     -0.0047       0.0059        -2.9986


### Passo 4 - Resíduos contra ajustado, contra `lag` e por região

O resíduo quantílico deve ter média 0 e dispersão 1 em qualquer faixa do ajustado (η̂ = logit π̂), do `lag` e em cada região. A curva vermelha é a média do resíduo em 20 faixas com o mesmo número de municípios: tendência na curva indica forma funcional errada (por exemplo, o coeficiente do `lag` ou um termo faltando); funil indica variância mal especificada. As correlações de Spearman e o desvio padrão por região (tabelas abaixo) resumem os gráficos.

In [18]:
eta = {k: np.log(PI[k] / (1 - PI[k])) for k in MODELOS_D}
x_lag = df[mapa["lag"]].to_numpy(float)
fig1 = G.residuos_contra(eta["bb_logit"], R, "logit do ajustado (η̂ de bb_logit, mesmo eixo em todos)", "Resíduo contra o ajustado")
fig2 = G.residuos_contra(x_lag, R, "lag = logit PT 2º turno 2022", "Resíduo contra o lag")
fig3 = G.caixa_regiao(R, reg_s, "Resíduos quantílicos por região")
for f, nm in [(fig1, "res_ajustado"), (fig2, "res_lag"), (fig3, "res_regiao")]:
    G.salvar(f, FIG / f"diag_{nm}.html"); f.show()
corr = pd.DataFrame({k: {"spearman_com_eta": stats.spearmanr(R[k], eta[k])[0], "spearman_com_lag": stats.spearmanr(R[k], x_lag)[0],
                         "corr_abs_r_eta": stats.spearmanr(np.abs(R[k]), eta[k])[0]} for k in MODELOS_D}).T
dp_reg = pd.DataFrame({k: pd.Series(R[k]).groupby(reg_s).std() for k in MODELOS_D}).T
med_reg = pd.DataFrame({k: pd.Series(R[k]).groupby(reg_s).mean() for k in MODELOS_D}).T
corr.rename_axis("modelo").to_csv(TAB / "diag_correlacoes.csv"); dp_reg.rename_axis("modelo").to_csv(TAB / "diag_dp_regiao.csv")
med_reg.rename_axis("modelo").to_csv(TAB / "diag_media_regiao.csv")
print(corr.round(3).to_string()); print("DP do resíduo por região:"); print(dp_reg.round(2).to_string())
print("média do resíduo por região:"); print(med_reg.round(2).to_string())

              spearman_com_eta  spearman_com_lag  corr_abs_r_eta
bin_logit               0.0110            0.0160          0.1000
quasi_reglag            0.0180            0.0230          0.0960
bb_logit                0.0110            0.0140          0.0980
bb_reglag               0.0140            0.0160          0.0910
bb_spline               0.0080            0.0100          0.0950
glmm_uf                -0.0030           -0.0010          0.0860
DP do resíduo por região:
              Centro-Oeste  Nordeste  Norte  Sudeste    Sul
bin_logit           3.4400    3.8700 3.9700   3.0300 2.9500
quasi_reglag        3.4800    3.8600 3.9200   3.0200 2.9400
bb_logit            0.9600    1.1900 1.4200   0.8000 0.9300
bb_reglag           0.9500    1.1900 1.4000   0.8000 0.9400
bb_spline           0.9600    1.1800 1.4100   0.8000 0.9400
glmm_uf             3.1100    3.6200 3.9000   2.9300 2.8100
média do resíduo por região:
              Centro-Oeste  Nordeste   Norte  Sudeste     Sul
bin_logi

### Passo 5 - Autocorrelação espacial dos resíduos

O pressuposto de independência entre municípios é o mais frágil: a EDA achou I de Moran = 0,896 para a proporção do PT e 0,542 para o *swing*. Calculamos o I de Moran dos resíduos quantílicos de cada modelo (esda, pesos de contiguidade rainha padronizados por linha, municípios-ilha ligados ao vizinho mais próximo, como no notebook 01; 999 permutações). Se os preditores absorvem a estrutura espacial, I ≈ 0. Se I > 0 permanece, municípios vizinhos erram na mesma direção; a informação efetiva é menor que N e **os erros-padrão e intervalos continuam pequenos demais** mesmo no beta-binomial. A estimativa pontual não fica necessariamente enviesada, mas o teste de significância e os intervalos ficam otimistas. A checagem honesta de desempenho é a validação cruzada *deixando uma UF de fora por vez* (notebook 03), que não deixa vizinhos no treino em que o teste foi avaliado, exceto na fronteira entre UFs.

O mapa de resíduos de `bb_reglag` é salvo só em HTML (`results/figures/diag_mapa_residuos_bb_reglag.html`, geometria simplificada a 0,05°), para não pesar o notebook.

In [19]:
import geopandas as gpd, libpysal, esda
from libpysal.weights import Queen, KNN
from libpysal.weights.util import attach_islands
geo = pd.read_parquet(ROOT / "data/processed/municipios_geo.parquet")
geo["id_municipio"] = geo["id_municipio"].astype(df["id_municipio"].dtype)
geo = geo.set_index("id_municipio").loc[df["id_municipio"]].reset_index()
gdf = gpd.GeoDataFrame(geo, geometry=gpd.GeoSeries.from_wkb(geo["geometry"]), crs="EPSG:4674")
w = Queen.from_dataframe(gdf, use_index=False)
w = attach_islands(w, KNN.from_dataframe(gdf, k=1)); w.transform = "r"
print("ilhas após ligação:", len(w.islands), "| vizinhos médios:", round(w.mean_neighbors, 2))
linhas = []
for k in MODELOS_D + ["bin_logit_pearson"]:
    r_k = R[k] if k in R else (y - n * PI["bin_logit"]) / np.sqrt(n * PI["bin_logit"] * (1 - PI["bin_logit"]))
    mi = esda.Moran(r_k, w, permutations=999)
    linhas.append({"modelo": k, "I": mi.I, "esperado": mi.EI, "z_sim": mi.z_sim, "p_sim": mi.p_sim, "dp_residuo": r_k.std(ddof=1)})
moran = pd.DataFrame(linhas); moran.to_csv(TAB / "diag_moran.csv", index=False)
print(moran.round(4).to_string(index=False))
fig = G.mapa_residuos(gdf, R["bb_reglag"], "Resíduos quantílicos de bb_reglag (limitados a ±4)")
G.salvar(fig, FIG / "diag_mapa_residuos_bb_reglag.html")
print("mapa salvo; tempo da Parte 10:", round(time.time() - t10), "s")

ilhas após ligação: 0 | vizinhos médios: 5.79


           modelo      I  esperado   z_sim  p_sim  dp_residuo
        bin_logit 0.3446   -0.0002 41.2086 0.0010      3.4269
     quasi_reglag 0.3517   -0.0002 43.3857 0.0010      3.4218
         bb_logit 0.3173   -0.0002 39.1149 0.0010      1.0336
        bb_reglag 0.3224   -0.0002 40.2169 0.0010      1.0326
        bb_spline 0.3169   -0.0002 40.4644 0.0010      1.0332
          glmm_uf 0.2856   -0.0002 34.8294 0.0010      3.2543
bin_logit_pearson 0.3344   -0.0002 40.7997 0.0010      4.6431


mapa salvo; tempo da Parte 10: 395 s


### Leitura da Parte 10: o que os diagnósticos permitem concluir

Nenhum modelo é escolhido aqui; o notebook 03 decide por validação cruzada. Aqui só dizemos quais pressupostos se sustentam. Números: tabelas `results/tables/diag_*.csv`.

**Distribuição da resposta.**
- `bin_logit`, `quasi_reglag` e `glmm_uf` **falham**: desvio padrão dos resíduos quantílicos 3,43, 3,42 e 3,25 (esperado 1), 52% a 55% dos resíduos fora de ±2 (esperado 4,55%) e 100% do half-normal fora do envelope. O quase-binomial corrige o erro-padrão por φ̂ = 21,5, mas não a forma da distribuição; o efeito aleatório de UF reduz a dispersão em apenas ~5%.
- Os três beta-binomiais (`bb_logit`, `bb_reglag`, `bb_spline`) **acertam a escala média** (desvio padrão 1,03; 4,9% a 5,0% fora de ±2) e a escolha de θ, mas **falham no formato**: assimetria -0,41, curtose em excesso 3,8 a 3,9 e 1,1% fora de ±3 (esperado 0,27%). No half-normal de `bb_reglag`, 89% dos pontos ficam abaixo da banda (corpo estreito demais) e 4,7% acima (caudas pesadas demais). O desvio padrão do resíduo varia por região, de 0,80 (Sudeste) a 1,40 (Norte), ou seja, **a dispersão θ não é constante**: o pressuposto de uma precisão única não se sustenta, e a evidência é de heterogeneidade regional da dispersão, não de falta de ajuste na média.

**Preditor linear.** Resíduos de todos os modelos têm correlação de Spearman ≤ 0,03 (em módulo) com o ajustado e com o `lag`, e médias regionais próximas de 0 nos beta-binomiais (|média| ≤ 0,02): não há tendência nem forma funcional visivelmente errada. `bb_spline` e `bb_reglag` quase não melhoram os resíduos frente a `bb_logit`.

**Pontos influentes.** Os 10 maiores valores de Cook são capitais e grandes cidades (Brasília e São Paulo concentram D = 5,1 e 3,9). No quase-binomial, retirá-los muda 22 dos 29 coeficientes em mais de 0,5 EP, até 8,4 EP (`bloco_outro`), com 2 trocas de sinal; os pesos proporcionais a nᵢ deixam poucos municípios dominar o ajuste. No beta-binomial a mudança máxima é 0,14 EP (nenhuma troca de sinal): o θ finito limita o peso das cidades grandes. **O beta-binomial é robusto a influentes; o quase-binomial não.**

**Independência espacial.** Falha em todos: I de Moran dos resíduos de 0,29 a 0,35 (p_sim = 0,001, o mínimo com 999 permutações). Os preditores (e a UF no GLMM, I = 0,29, o menor) explicam parte da estrutura espacial da EDA (I = 0,90 na proporção), mas sobra muita. Consequência: **os erros-padrão e intervalos de todos os modelos continuam pequenos demais** e testes de significância dos coeficientes não devem ser lidos de forma literal. A estimativa pontual de π não é necessariamente enviesada; o desempenho preditivo honesto sai da validação deixando uma UF de fora (notebook 03).

**Resumo por modelo.**

| modelo | distribuição | preditor | influentes | independência |
|---|---|---|---|---|
| `bin_logit` | falha | ok | não avaliado (usa-se o quase) | falha |
| `quasi_reglag` | falha na forma (EP corrigido por φ) | ok | falha | falha |
| `bb_logit`, `bb_spline` | escala ok, formato e θ constante falham | ok | ok (modelo irmão `bb_reglag`) | falha |
| `bb_reglag` | escala ok, formato e θ constante falham | ok | ok | falha |
| `glmm_uf` | falha | ok | não avaliado | falha (menor I) |

Possíveis remédios, que ficam para a validação ou para trabalho futuro: dispersão por região (θ função da região), termo espacial ou erros-padrão robustos a agrupamento por UF.